###  This is to work on assginment 
In this notebook I will try to implement the DQN presneted in paper . to learn Key objectives are a

[Y] environment setup.

[] preprocessing of image for CNN

[] CNN architecture presented in paper

[] training CNN for making action from image 

[] different Q learning options 

[] Metrics to measure the RL.




# setting the Environment


In [ ]:
%pip install ale-py

In [ ]:
%pip install opencv-python
%pip install gymnasium

In [ ]:
import torch 
import torch.nn as nn

class DQN(nn.Module):

    def __init__(self, in_channels: int = 4, num_actions: int = 4):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size = 8, stride = 4)
        self.conv2 = nn.Conv2d(16, 32, kernel_size = 4, stride = 2)
        self.relu = nn.ReLU()

        #faltting the size
        self._flatten_size = self._infer_flatten_size(in_channels)

        self.fc1 = nn.Linear(self._flatten_size, 256)
        self.out = nn.Linear(256, num_actions)

        self._init_weights()

    def _infer_flatten_size(self,in_channels: int) -> int:
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels,84,84)
            x = self.relu(self.conv1(dummy))
            x = self.relu(self.conv2(x))
            return x.numel()

    def _init_weights(self):

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor : 

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.flatten(start_dim=1)
        x = self.relu(self.fc1(x))
        q_values = self.out(x)
        return q_values


In [ ]:
from collections import deque
import numpy as np
import cv2
import torch

class AtariPreprocessor:

    RAW_HEIGHT, RAW_WIDTH = 210, 160
    RESIZE_HEIGHT , RESIZE_WIDTH = 110, 84
    CROP_SIZE = 84
    STACK_SIZE = 4

    def __init__(self):
        self.frame_stack = deque(maxlen = self.STACK_SIZE)

    @staticmethod
    def to_grayscale(frame_rgb: np.ndarray) -> np.ndarray:
        assert frame_rgb.ndim ==3 and frame_rgb.shape[2] ==3, (f"Expected an HXWx3 frame got shape {frame_rgb.shape}")
        return cv2.cvtColor(frame_rgb,cv2.COLOR_RGB2GRAY)

    def downsample(self, frame_gray: np.ndarray) -> np.ndarray:
        return cv2.resize(frame_gray, (self.RESIZE_WIDTH, self.RESIZE_HEIGHT), interpolation = cv2.INTER_LINEAR)

    def crop(self, frame_110x84: np.ndarray) ->np.ndarray:
        top = self.RESIZE_HEIGHT - self.CROP_SIZE # 110-84 =26 
        return frame_110x84[top:top+self.CROP_SIZE, 0:self.CROP_SIZE]

    def preprocess_single_frame(self,frame_rgb: np.ndarray) -> np.ndarray:
        gray = self.to_grayscale(frame_rgb)
        resized = self.downsample(gray)
        cropped = self.crop(resized)
        assert cropped.shape == (self.CROP_SIZE, self.CROP_SIZE)
        return cropped

    def step(self, frame_rgb: np.ndarray) -> np.ndarray:
        processed = self.preprocess_single_frame(frame_rgb)
        self.frame_stack.append(processed)

        while len(self.frame_stack)< self.STACK_SIZE:
            self.frame_stack.append(processed)
        
        stacked = np.stack(self.frame_stack, axis=-1) # (84,84,4)
        assert stacked.shape == (self.CROP_SIZE, self.CROP_SIZE, self.STACK_SIZE)
        return stacked

    def reset(self):
        self.frame_stack.clear()

    def to_model_input(stacked_frame):
        x = stacked_frame.astype("flaot32")/255.0
        x = torch.from_numpy(x).premute(2,0,1)
        return x.unsquuze(0)
        
        

In [ ]:
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import ale_py
import gymnasium as gym

USE_TOY_ENV = False
GAME_ID = "ALE/Pong-v5"

def make_env():
    gym.register_envs(ale_py)
    return gym.make(GAME_ID, render_mode = "rgb_array")

# Hyper Parameters
NUM_EPISODES = 80

REPLAY_CAPACITY = 20000

MIN_REPLAY_BEFORE_TRAIN = 1000

BATCH_SIZE = 32

GAMMA = 0.99

EPS_START = 1.0

EPS_END = 0.1

EPS_DECAY_STEPS = 20000

LEARNING_RATE = 2.5e-4

TARGET_EVAL_STATES = 64

LOG_EVERY = 5 

CLIP_REWARDS = True

class ReplayMemory:

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen = capacity)

    def push(self, phi,action, reward,next_phi,done):
        self.buffer.append((phi,action,reward,next_phi,done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer,batch_size)
        phis, actions,rewards,next_phis,dones = zip(*batch)
        return (
            np.stack(phis),
            np.array(actions, dtype = np.int64),
            np.array(rewards, dtype = np.float32),
            np.stack(next_phis),
            np.array(dones, dtype = np.float32),
        )

    def __len__(self):
        return len(self.buffer)

def to_tensor_batch(phi_batch_hwcN: np.ndarray) -> torch.Tensor:
        x = phi_batch_hwcN.astype(np.float32)/255.0
        x = torch.from_numpy(x).permute(0,3,1,2)
        return x

def clip_reward(reward: float) -> float:
    if reward >0 :
        return 1.0
    if reward <0:
        return -1.0
    return 0.0

def epsilon_by_step(step: int) ->float:
    frac = min(1.0, step/EPS_DECAY_STEPS)
    return EPS_START + frac* (EPS_END - EPS_START)

def select_action(net: DQN, phi_hwc: np.ndarray , epsilon: float, num_actions: int) -> int:

    if random.random() <epsilon:
        return random.randrange(num_actions)
    with torch.no_grad():
        x = to_tensor_batch(phi_hwc[None, ...])
        q_values = net(x)
        return int(torch.argmax(q_values, dim=1).item())

def collect_eval_states(env, preprocessor, n_states: int):
    states = []
    obs, info = env.reset()
    preprocessor.reset()
    phi = preprocessor.step(obs)
    states.append(phi)
    while len(states) < n_states:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        phi = preprocessor.step(obs)
        states.append(phi)
        if terminated or truncated:
            obs, info = env.reset()
            preprocessor.reset()
            phi = preprocessor.step(obs)
    return np.stack(states[:n_states])

def compute_avg_max_q(net: DQN, eval_states: np.ndarray) -> float:
    with torch.no_grad():
        x = to_tensor_batch(eval_states)
        q_values = net(x)
        max_q = q_values.max(dim=1).values
        return float(max_q.mean().item())
    


def train():
    env = make_env()
    num_actions = env.action_space.n

    preprocessor = AtariPreprocessor()
    net = DQN(in_channels= 4, num_actions =num_actions)
    optimizer = optim.RMSprop(net.parameters(), lr = LEARNING_RATE)
    loss_fn = nn.MSELoss()

    replay = ReplayMemory(REPLAY_CAPACITY)
    print("Collecting a foxed held out evaluations set random policy")
    eval_states = collect_eval_states(env,preprocessor, TARGET_EVAL_STATES)

    episode_rewards = []
    avg_q_history = []
    loss_history = []

    global_step = 0

    for episode in range(1, NUM_EPISODES+1):
        obs, info = env.reset()
        preprocessor.reset()
        phi = preprocessor.step(obs)

        episode_reward = 0.0
        done = False

        while not done:
            epsilon = epsilon_by_step(global_step)
            action = select_action(net,phi, epsilon, num_actions)

            obs, reward, terminated, turncated, info = env.step(action)
            done = terminated or turncated
            next_phi = preprocessor.step(obs)
            train_reward = clip_reward(reward) if CLIP_REWARDS else reward

            # algo store transaction (phi_t, a_t, r_t, phi_t+1)
            replay.push(phi, action, train_reward, next_phi, float(terminated))

            phi = next_phi
            episode_reward += reward # actual score
            global_step +=1

            # algo 2 sample random minibatch and do one SGD step

            if len(replay) >= max(MIN_REPLAY_BEFORE_TRAIN, BATCH_SIZE):
                phis, actions, rewards,next_phis, dones = replay.sample(BATCH_SIZE)

                states_t = to_tensor_batch(phis)
                next_states_t = to_tensor_batch(next_phis)
                actions_t = torch.from_numpy(actions).long()
                rewards_t = torch.from_numpy(rewards).float()
                dones_t = torch.from_numpy(dones).float()


                # no stablization 
                with torch.no_grad():
                    next_q = net(next_states_t)
                    max_next_q = next_q.max(dim=1).values
                    targets = rewards_t +GAMMA*max_next_q*(1.0-dones_t)

                current_q_all = net(states_t)
                current_q = current_q_all.gather(1, actions_t.unsqueeze(1)).squeeze(1)

                loss = loss_fn(current_q, targets)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                loss_history.append(loss.item())

        episode_rewards.append(episode_reward)
        avg_q = compute_avg_max_q(net, eval_states)
        avg_q_history.append(avg_q)


        # logging
        if episode % LOG_EVERY ==0 :
            recent_reward = np.mean(episode_rewards[-LOG_EVERY])
            recent_loss = np.mean(loss_history[-200:])if loss_history else float("naan")
            print(f"Episode {episode:4d} | avg reward (last {LOG_EVERY}) = {recent_reward:+.2f}| ")
            

    env.close()
    return net, episode_rewards,avg_q_history, loss_history

    
                


            

In [ ]:
net, episode_rewards,avg_q_history, loss_history = train()

tag = GAME_ID
weights_path = f"dqn_{tag}.pt"
history_path = f"dqn_training_history_{tag}.npz"

torch.save(net.state_dict(),weight_paths)
np.savez(history_path, epsiode_rewards = np.array(episode_rewards), avg_q_history=np.array(avg_q_history),loss_history=np.array(loss_history))

print(f"\n Saved trained weights to {weights_path}")

